# Predict Kaggle `test.csv` → `submission.csv`
1. Re-use the validated recipe from `train.ipynb` (same features, params, and number of trees).
2. Refit cleaning + model on **all** labelled rows: the held-out score has already been measured.
3. Clean Kaggle's `test.csv` with stats learned from train (never from test), predict, and write the file.

In [1]:
%run train.ipynb

(5215, 14) (1739, 14) (1739, 14)
Learned from train split: {'age': np.float64(27.0), 'homeplanet': 'Earth', 'cryosleep': False, 'destination': 'TRAPPIST-1e', 'vip': False}
Missing after cleaning: 0 0 0


Selected Numeric Features: ['roomservice', 'spa', 'vrdeck']
Selected Categorical Features: ['cryosleep']
['roomservice', 'spa', 'vrdeck', 'cryosleep']
(5215, 4) (1739, 4) (1739, 4)
(5215,) (1739,) (1739,)


0:	test: 0.8017422	best: 0.8017422 (0)	total: 134ms	remaining: 4m 28s


200:	test: 0.8117609	best: 0.8119276 (152)	total: 454ms	remaining: 4.07s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8119275703
bestIteration = 152

Shrink model to first 153 iterations.
train      acc=0.7745  f1=0.7860  auc=0.8308
validation acc=0.7602  f1=0.7732  auc=0.8119
test       acc=0.7746  f1=0.7839  auc=0.8200


## Refit on all labelled data
Early stopping can't run without a validation set, so use the best iteration found in `train.ipynb`.

In [2]:
full_stats = fit_cleaner(df)
full_clean = apply_cleaner(df, full_stats)
X_full = to_model_input(full_clean)
y_full = full_clean[target].astype(int)

n_trees = model.get_best_iteration() + 1
final_model = CatBoostClassifier(**CATBOOST_PARAMS, iterations=n_trees, verbose=0)
final_model.fit(X_full, y_full, cat_features=selected_cat_features)

print("Trees:", n_trees, "| rows:", len(X_full), "| features:", selected_features)
print("Fill values learned from all of train.csv:", full_stats)

Trees: 153 | rows: 8693 | features: ['roomservice', 'spa', 'vrdeck', 'cryosleep']
Fill values learned from all of train.csv: {'age': np.float64(27.0), 'homeplanet': 'Earth', 'cryosleep': False, 'destination': 'TRAPPIST-1e', 'vip': False}


## Clean Kaggle test with the train stats and predict

In [3]:
kaggle_test = standardize_columns(pd.read_csv("./data/raw/test.csv"))
kaggle_clean = apply_cleaner(kaggle_test, full_stats)  # stats from train, not test
X_kaggle = to_model_input(kaggle_clean)

kaggle_pred = final_model.predict(X_kaggle).astype(int).ravel()
print(X_kaggle.shape, "missing:", int(X_kaggle.isna().sum().sum()))
print("Predicted transported rate:", kaggle_pred.mean().round(4),
      "| train rate:", y_full.mean().round(4))

(4277, 4) missing: 0
Predicted transported rate: 0.5569 | train rate: 0.5036


## Write `submission.csv`
Kaggle wants `PassengerId,Transported` with `True`/`False`, one row per test passenger.

In [4]:
submission = pd.DataFrame({
    "PassengerId": kaggle_test["passengerid"],
    "Transported": kaggle_pred.astype(bool),
})

sample = pd.read_csv("./data/raw/sample_submission.csv")
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(sample)
assert submission["PassengerId"].equals(sample["PassengerId"])
assert submission["Transported"].notna().all()

submission.to_csv("./submission.csv", index=False)
print("Saved ./submission.csv", submission.shape)
submission.head()

Saved ./submission.csv (4277, 2)


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,False
4,0023_01,True
